In [ ]:
import cv2
import time
import numpy as np
import os

# Paths to Haar cascade XML files
cascade_path = "/opt/anaconda3/envs/cv-env/share/opencv4/haarcascades/haarcascade_frontalface_default.xml"
eye_cascade_path = "/opt/anaconda3/envs/cv-env/share/opencv4/haarcascades/haarcascade_eye.xml"

# Load classifiers
detector = cv2.CascadeClassifier(cascade_path)
eye_cascade = cv2.CascadeClassifier(eye_cascade_path)

def detect_pupil(eye_region):
    """Detect the pupil in the eye region using thresholding and contour detection."""
    gray = cv2.cvtColor(eye_region, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (7, 7), 0)
    _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY_INV)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        (x, y), radius = cv2.minEnclosingCircle(largest_contour)
        return int(x), int(y), int(radius)
    return None

# Open webcam
cap = cv2.VideoCapture(0)
prev_time = time.time()
frame_count = 0

# Data for measurement noise estimation
pupil_positions = []
errors = []

# Kalman Filter Initialization
state = np.zeros((4, 1))  # [x, y, vx, vy]
P = np.eye(4) * 500  # Initial state covariance
dt = 1 / 30  # Approximate time between frames

F = np.array([[1, 0, dt, 0],
              [0, 1, 0, dt],
              [0, 0, 1, 0],
              [0, 0, 0, 1]])

H = np.array([[1, 0, 0, 0],
              [0, 1, 0, 0]])

Q = np.eye(4) * 0.1  # Process noise covariance

# Initial measurement noise covariance (will update after collecting data)
R = np.eye(2) * 50  

I = np.eye(4)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector.detectMultiScale(gray_frame, scaleFactor=1.3, minNeighbors=5, minSize=(50, 50))
    
    for (x, y, w, h) in faces:
        face_region = frame[y:y+h, x:x+w]
        eyes = eye_cascade.detectMultiScale(face_region, scaleFactor=1.1, minNeighbors=5, minSize=(20, 20))
        
        for (ex, ey, ew, eh) in eyes:
            eye_roi = face_region[ey:ey+eh, ex:ex+ew]
            pupil_center = detect_pupil(eye_roi)
            
            # Kalman Predict
            state = F @ state
            P = F @ P @ F.T + Q

            if pupil_center:
                px, py, pr = pupil_center
                pupil_positions.append((px, py))
                
                measurement = np.array([[px], [py]])

                # Kalman Update
                y_k = measurement - H @ state
                S = H @ P @ H.T + R
                K = P @ H.T @ np.linalg.inv(S)
                state = state + K @ y_k
                P = (I - K @ H) @ P

                # Calculate error
                pred_x, pred_y = int(state[0, 0]), int(state[1, 0])
                error = np.sqrt((px - pred_x)**2 + (py - pred_y)**2)
                errors.append(error)

                # Draw measurement (green circle)
                cv2.circle(eye_roi, (px, py), pr, (0, 255, 0), 2)
                
            # Draw prediction (red dot)
            pred_x, pred_y = int(state[0, 0]), int(state[1, 0])
            cv2.circle(eye_roi, (pred_x, pred_y), 5, (0, 0, 255), -1)

            # Draw eye rectangle
            cv2.rectangle(face_region, (ex, ey), (ex+ew, ey+eh), (255, 0, 0), 2)

    # After collecting enough data, update R
    if len(pupil_positions) > 50:
        positions = np.array(pupil_positions)
        variance_x = np.var(positions[:, 0])
        variance_y = np.var(positions[:, 1])
        R = np.array([[variance_x, 0],
                      [0, variance_y]])
        print("Updated measurement noise covariance matrix R:")
        print(R)
        pupil_positions.clear()  # clear to prevent overflow

    # Calculate frame rate
    frame_count += 1
    current_time = time.time()
    fps = frame_count / (current_time - prev_time)

    # Display average tracking error
    if errors:
        avg_error = np.mean(errors)
        cv2.putText(frame, f"Avg Error: {avg_error:.2f}px", (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    # Display FPS
    cv2.putText(frame, f"FPS: {fps:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Eye Tracking with Kalman Filter", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Final summary after the loop ends
if errors:
    print(f"\nFinal average tracking error: {np.mean(errors):.2f} pixels")
else:
    print("\nNo tracking error data collected.")

2025-04-10 10:25:17.390 python[52726:28241997] +[IMKClient subclass]: chose IMKClient_Modern
2025-04-10 10:25:17.390 python[52726:28241997] +[IMKInputSession subclass]: chose IMKInputSession_Modern


Updated measurement noise covariance matrix R:
[[455.94193787   0.        ]
 [  0.         201.82359467]]
Updated measurement noise covariance matrix R:
[[150.30758277   0.        ]
 [  0.         184.71698113]]
Updated measurement noise covariance matrix R:
[[350.87980769   0.        ]
 [  0.         450.36686391]]
Updated measurement noise covariance matrix R:
[[344.32015306   0.        ]
 [  0.         401.83896684]]
Updated measurement noise covariance matrix R:
[[735.9750801   0.       ]
 [  0.        578.8244927]]
Updated measurement noise covariance matrix R:
[[391.43098808   0.        ]
 [  0.         329.33333333]]
Updated measurement noise covariance matrix R:
[[48.92502884  0.        ]
 [ 0.         57.56862745]]
Updated measurement noise covariance matrix R:
[[167.41253364   0.        ]
 [  0.          80.59823145]]
Updated measurement noise covariance matrix R:
[[95.5443787   0.        ]
 [ 0.         28.11649408]]
Updated measurement noise covariance matrix R:
[[72.159763

: 

In [ ]:
# The final average tracking error is approximately 16.42 pixels. This shows that while the filter improves the stability of the detection, there is still some noise likely due to lighting variation, rapid eye movements, and detection inaccuracy. However, using the Kalman filter has successfully smoothed out fluctuations in measurement, and the error is within an acceptable range for real-time applications.